In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: YAlO3+Al2O3, SPODI

This example demonstrates a staged two-phase Rietveld refinement of
yttrium aluminium perovskite YAlO3 (or YAP) with a small Al2O3
impurity using constant wavelength neutron powder diffraction data
measured at 3 K on SPODI at MLZ.

The workflow defines both structures, configures the experiment, and
refines the cell, scale, profile, background, and atom parameters of
both phases in stages.

## 🛠️ Import Library

In [ ]:
import easydiffraction as edi

## 📦 Define Project

### Create Project

In [ ]:
project = edi.Project(
    name='yap_3k',
    description='Two-phase YAlO3 and Al2O3 refinement using 3 K data from SPODI at MLZ.',
)

### Save Initial Project

In [ ]:
project.save_as(dir_path='projects/refine-yap-3k')

## 🧩 Define Structures

### Create Structure 1: YAlO3

Preserve the orthorhombic Pbnm setting used in FullProf. In
EasyDiffraction this is represented by the standard space-group
symbol `P n m a` with coordinate-system code `cab`. The cell axes and
atom coordinates below therefore stay in the original Pbnm setting.

FullProf's PCR occupancies include the site multiplicity divided by
the general-position multiplicity. Here each atom site is fully
occupied: the PCR values 0.5 for Y, Al, and O1, and 1.0 for O2, all
become an occupancy of 1.0. Displacement parameters are entered as
Biso, matching the PCR file.

In [ ]:
yap_cif = """
data_yap

_cell.length_a 5.18
_cell.length_b 5.33
_cell.length_c 7.37
_cell.angle_alpha 90.
_cell.angle_beta 90.
_cell.angle_gamma 90.

_space_group.name_h_m "P n m a"
_space_group.coord_system_code cab

loop_
_atom_site.id
_atom_site.type_symbol
_atom_site.fract_x
_atom_site.fract_y
_atom_site.fract_z
_atom_site.occupancy
_atom_site.adp_iso
_atom_site.adp_type
Y  Y   0.0100  0.5500 0.2500 1.0 0.12 Biso
Al Al  0.0000  0.0000 0.0000 1.0 0.13 Biso
O1 O  -0.0800 -0.0200 0.2500 1.0 0.06 Biso
O2 O   0.2000  0.2900 0.0400 1.0 0.14 Biso
"""

In [ ]:
project.structures.add_from_cif_str(yap_cif)

In [ ]:
yap = project.structures['yap']

### Display Structure 1: YAlO3

In [ ]:
yap.show_as_text()

In [ ]:
project.display.structure(struct_name='yap')

### Create Structure 2: Al2O3

Define the corundum impurity in the hexagonal setting of R-3c. The
FullProf PCR occupancies of 2/3 for Al and 1 for O also describe fully
occupied sites. Refine its two independent cell lengths, Al z and O x
coordinates, and both Biso values, as specified by the PCR codewords.
The PCR contains a negative Al Biso. EasyDiffraction requires a
nonnegative input value, so start this parameter at 0.1 Å² and refine
it alongside O Biso.

In [ ]:
alumina = edi.StructureFactory.from_scratch(name='alumina')

#### Set Space Group

In [ ]:
alumina.space_group.name_h_m = 'R -3 c'
alumina.space_group.coord_system_code = 'h'

#### Set Unit Cell

In [ ]:
alumina.cell.length_a = 4.75
alumina.cell.length_c = 12.95

#### Set Atom Sites

In [ ]:
alumina.atom_sites.create(
    id='Al1',
    type_symbol='Al',
    fract_x=0.0,
    fract_y=0.0,
    fract_z=0.33351,
    occupancy=1.0,
    adp_type='Biso',
    adp_iso=0.1,
)
alumina.atom_sites.create(
    id='O1',
    type_symbol='O',
    fract_x=0.3503,
    fract_y=0.0,
    fract_z=0.25,
    occupancy=1.0,
    adp_type='Biso',
    adp_iso=1.22884,
)

In [ ]:
project.structures.add(alumina)

### Display Structure 2: Al2O3

In [ ]:
alumina.show_as_text()

In [ ]:
project.display.structure(struct_name='alumina')

## 🔬 Define Experiment

### Download Measured Data

Download the YAlO3 + Al2O3 pattern from the EasyDiffraction online
data repository. The three columns contain 2-theta in degrees,
intensity, and its standard uncertainty. They are copied from the
original SPODI dataset without changing the measured values.

In [ ]:
data_path = edi.download_data('meas-yap-spodi', destination='data')

In [ ]:
project.experiments.add_from_data_path(
    name='yap_3k',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

In [ ]:
expt = project.experiments['yap_3k']

### Set Instrument

Use the neutron wavelength reported for the SPODI dataset and start
with zero 2-theta offset.

In [ ]:
expt.instrument.setup_wavelength = 1.54816
expt.instrument.calib_twotheta_offset = 0.0

### Set Peak Profile

Use a pseudo-Voigt profile with Bérar-Baldinozzi asymmetry.

In [ ]:
expt.peak.show_supported()

In [ ]:
expt.peak.type = 'pseudo-voigt + berar-baldinozzi asymmetry'

In [ ]:
expt.peak.broad_gauss_u = 0.04
expt.peak.broad_gauss_v = -0.05
expt.peak.broad_gauss_w = 0.10
expt.peak.broad_lorentz_x = 0.0
expt.peak.broad_lorentz_y = 0.01

In [ ]:
expt.peak.asym_beba_a0 = 0.0
expt.peak.asym_beba_b0 = 0.0
expt.peak.asym_beba_a1 = 0.0
expt.peak.asym_beba_b1 = 0.0

In [ ]:
expt.peak.cutoff_fwhm = 8.0

### Set Absorption

Apply the cylindrical-sample Hewat correction with the absorption
radius product from FullProf.

In [ ]:
expt.absorption.type = 'cylinder-hewat'
expt.absorption.mu_r = 0.0221

### Set Excluded Regions

In [ ]:
expt.excluded_regions.create(id='1', start=0.0, end=4.0)
expt.excluded_regions.create(id='2', start=153.95, end=180.0)

### Set Background

Estimate the initial line-segment background from the measured pattern.
This first estimate does not use a calculated structural model.

In [ ]:
expt.background.auto_estimate(use_model=False)

In [ ]:
expt.background.show()

### Set Linked Structures

Give each phase an independent scale factor. Scale factors are fitted
intensity multipliers, rather than phase weight fractions.

In [ ]:
expt.linked_structures.create(structure_id='yap', scale=30)
expt.linked_structures.create(structure_id='alumina', scale=0.1)

In [ ]:
expt.show_as_text()

## 🚀 Perform Analysis

### Display Initial Pattern

In [ ]:
project.display.pattern(expt_name='yap_3k')

In [ ]:
project.display.pattern(expt_name='yap_3k', x_min=134, x_max=146)

### Select Calculator

In [ ]:
expt.calculator.show_supported()

In [ ]:
expt.calculator.type = 'cryspy'

### Select Minimizer

In [ ]:
project.analysis.minimizer.show_supported()

In [ ]:
project.analysis.minimizer.type = 'bumps (lm)'

In [ ]:
project.analysis.minimizer.max_iterations = 500
project.analysis.minimizer.chi_square_change_tolerance = 1e-2

### Perform Fit 1/3: Cell, Scale, and Background

First refine the independent cell lengths of both phases, both phase
scales, the instrument zero offset, and the automatically estimated
background intensities. Hexagonal symmetry couples the Al2O3 b length
to a, leaving only a and c independent.

In [ ]:
yap.cell.length_a.free = True
yap.cell.length_b.free = True
yap.cell.length_c.free = True

alumina.cell.length_a.free = True
alumina.cell.length_c.free = True

expt.linked_structures['yap'].scale.free = True
expt.linked_structures['alumina'].scale.free = True

expt.instrument.calib_twotheta_offset.free = True

for point in expt.background:
    point.intensity.free = True

In [ ]:
project.display.parameters.free()

In [ ]:
project.analysis.fit()

In [ ]:
project.display.fit.results()

### Perform Fit 2/3: Peak Profile

Add the Gaussian and Lorentzian broadening and asymmetry parameters
to the refinement. The background intensities remain free.

In [ ]:
expt.peak.broad_gauss_u.free = True
expt.peak.broad_gauss_v.free = True
expt.peak.broad_gauss_w.free = True
expt.peak.broad_lorentz_y.free = True

expt.peak.asym_beba_a0.free = True
# expt.peak.asym_beba_b0.free = True
# expt.peak.asym_beba_a1.free = True
expt.peak.asym_beba_b1.free = True

In [ ]:
project.display.parameters.free()

In [ ]:
project.analysis.fit()

In [ ]:
project.display.fit.results()

### Perform Fit 3/3: Model-Guided Background and Atom Parameters

Replace the initial background with a new estimate based on the fitted
peak model. Automatically generated points are fixed by default, so
mark their intensities free before fitting them with the atom parameters.

In [ ]:
expt.background.auto_estimate(use_model=True)

In [ ]:
expt.background.show()

In [ ]:
for point in expt.background:
    point.intensity.free = True

Refine the independent Y and O coordinates in Pbnm and the
isotropic displacement parameters of both phases. Symmetry keeps Y
and O1 in YAlO3 at z = 1/4 and Al at the origin. For Al2O3, refine
Al1 z and O1 x. Occupancies remain fixed
at 1.0, and all coordinates fixed by symmetry remain fixed.

In [ ]:
yap.atom_sites['Y'].fract_x.free = True
yap.atom_sites['Y'].fract_y.free = True
yap.atom_sites['O1'].fract_x.free = True
yap.atom_sites['O1'].fract_y.free = True
yap.atom_sites['O2'].fract_x.free = True
yap.atom_sites['O2'].fract_y.free = True
yap.atom_sites['O2'].fract_z.free = True

alumina.atom_sites['Al1'].fract_z.free = True
alumina.atom_sites['O1'].fract_x.free = True

for structure in (yap, alumina):
    for atom in structure.atom_sites:
        atom.adp_iso.free = True

In [ ]:
project.display.parameters.free()

In [ ]:
project.analysis.fit()

### Inspect Results

Review the fit statistics, refined parameters, and correlations.
Inspect the full pattern and a closer view with impurity reflections.

In [ ]:
project.display.fit.results()

In [ ]:
project.display.fit.correlations()

In [ ]:
project.display.pattern(expt_name='yap_3k')

In [ ]:
project.display.pattern(expt_name='yap_3k', x_min=134, x_max=146)

## 💾 Save Project

Save the refined model and analysis results in the project directory.

In [ ]:
project.save()